In [2]:
import sys
from pathlib import Path

print("Python:", sys.executable)
print("Working directory:", Path.cwd())

Python: c:\Users\megdo\Desktop\underwater-object-detection\venv\Scripts\python.exe
Working directory: c:\Users\megdo\Desktop\underwater-object-detection\notebooks


In [ ]:
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd().parent # set the project root directory to the parent of the current working directory

if not (PROJECT_ROOT / "data").exists(): # check if the data directory exists in the project root
    PROJECT_ROOT = Path.cwd()

ANNOTATION_DIR = ( # set the annotation directory path to the processed DUO_coco annotations directory within the data directory of the project root
    PROJECT_ROOT
    / "data"
    / "processed"
    / "DUO_coco"
    / "annotations"
)

SPLIT_DIR = PROJECT_ROOT / "data" / "splits" # set the split directory path to the splits directory within the data directory of the project root

clean_train_path = ANNOTATION_DIR / "instances_train_clean.json" # set the clean training JSON path to the instances_train_clean.json file within the annotation directory
clean_test_path = ANNOTATION_DIR / "instances_test_clean.json" # set the clean test JSON path to the instances_test_clean.json file within the annotation directory

print("Project root:", PROJECT_ROOT.resolve())
print("Clean training JSON found:", clean_train_path.exists())
print("Clean test JSON found:", clean_test_path.exists())
print("Train split found:", (SPLIT_DIR / "train.txt").exists())
print("Validation split found:", (SPLIT_DIR / "val.txt").exists())

Project root: C:\Users\megdo\Desktop\underwater-object-detection
Clean training JSON found: True
Clean test JSON found: True
Train split found: True
Validation split found: True


In [ ]:
with clean_train_path.open("r", encoding="utf-8") as file: # open the clean training JSON file in read mode with UTF-8 encoding
    clean_train_data = json.load(file)

with clean_test_path.open("r", encoding="utf-8") as file: # open the clean test JSON file in read mode with UTF-8 encoding
    clean_test_data = json.load(file)

train_filenames = { #   create a set of training filenames by reading the train.txt file in the splits directory, stripping whitespace from each line, and including only non-empty lines
    line.strip()
    for line in (SPLIT_DIR / "train.txt").read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
}

val_filenames = { #   create a set of validation filenames by reading the val.txt file in the splits directory, stripping whitespace from each line, and including only non-empty lines
    line.strip()
    for line in (SPLIT_DIR / "val.txt").read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
}

print("Training filenames:", len(train_filenames))
print("Validation filenames:", len(val_filenames))
print("Official test images:", len(clean_test_data["images"]))

Training filenames: 5336
Validation filenames: 1335
Official test images: 1111


In [ ]:
def create_coco_subset(coco_data, selected_filenames): # define a function to create a COCO subset from the given COCO data and selected filenames
    selected_images = [
        image
        for image in coco_data["images"]
        if image["file_name"] in selected_filenames
    ]

    selected_image_ids = { # create a set of selected image IDs from the selected images
        image["id"]
        for image in selected_images
    }

    selected_annotations = [ #  create a list of selected annotations from the COCO data, including only those annotations whose image ID is in the set of selected image IDs
        annotation
        for annotation in coco_data["annotations"]
        if annotation["image_id"] in selected_image_ids
    ]

    subset = { # create a dictionary representing the COCO subset, including the selected images, selected annotations, and categories from the original COCO data
        "images": selected_images,
        "annotations": selected_annotations,
        "categories": coco_data["categories"],
    }

    if "info" in coco_data: # check if the info key exists in the original COCO data and, if so, include it in the subset
        subset["info"] = coco_data["info"]

    if "licenses" in coco_data: # check if the licenses key exists in the original COCO data and, if so, include it in the subset
        subset["licenses"] = coco_data["licenses"]

    return subset # return the created COCO subset


train_split_data = create_coco_subset( # create the training split data by calling the create_coco_subset function with the clean training data and training filenames
    clean_train_data,
    train_filenames,
)

val_split_data = create_coco_subset( # create the validation split data by calling the create_coco_subset function with the clean training data and validation filenames

    clean_train_data,
    val_filenames,
)

print(
    "Training split:",
    len(train_split_data["images"]),
    "images and",
    len(train_split_data["annotations"]),
    "annotations",
)

print(
    "Validation split:",
    len(val_split_data["images"]),
    "images and",
    len(val_split_data["annotations"]),
    "annotations",
)

Training split: 5336 images and 51500 annotations
Validation split: 1335 images and 12497 annotations


In [ ]:
train_split_path = ( # set the training split JSON path to the instances_train_split.json file within the
    ANNOTATION_DIR
    / "instances_train_split.json"
)

val_split_path = ( # set the validation split JSON path to the instances_val_split.json file within the
    ANNOTATION_DIR
    / "instances_val_split.json"
)

with train_split_path.open("w", encoding="utf-8") as file: # open the training split JSON file in write mode with UTF-8 encoding
    json.dump(train_split_data, file)

with val_split_path.open("w", encoding="utf-8") as file: # open the validation split JSON file in write mode with UTF-8 encoding
    json.dump(val_split_data, file)

print("Saved:", train_split_path)
print("Saved:", val_split_path)

Saved: c:\Users\megdo\Desktop\underwater-object-detection\data\processed\DUO_coco\annotations\instances_train_split.json
Saved: c:\Users\megdo\Desktop\underwater-object-detection\data\processed\DUO_coco\annotations\instances_val_split.json


In [ ]:
#check there is no overlap between train and val splits
train_names_check = {
    image["file_name"]
    for image in train_split_data["images"]
}

val_names_check = { # create a set of validation image filenames by extracting the "file_name" from each image in the validation split data
    image["file_name"]
    for image in val_split_data["images"]
}

overlap = train_names_check & val_names_check # compute the intersection of the training and validation image filename sets to find any overlapping filenames

print("Training images:", len(train_names_check))
print("Validation images:", len(val_names_check))
print("Overlapping filenames:", len(overlap))

Training images: 5336
Validation images: 1335
Overlapping filenames: 0


In [ ]:
import sys # import the sys module to access system-specific parameters and functions
import torch # import the torch module for PyTorch functionalities
import torchvision # import the torchvision module for computer vision functionalities in PyTorch

print("Python:", sys.executable)
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())

Python: c:\Users\megdo\Desktop\underwater-object-detection\venv\Scripts\python.exe
PyTorch version: 2.13.0+cpu
Torchvision version: 0.28.0+cpu
CUDA available: False


In [ ]:
TRAIN_IMAGE_DIR = PROJECT_ROOT / "data" / "raw" / "DUO" / "images" / "train" # set the training image directory path to the train directory within the images directory of the raw DUO data in the data directory of the project root
TEST_IMAGE_DIR = PROJECT_ROOT / "data" / "raw" / "DUO" / "images" / "test" # set the test image directory path to the test directory within the images directory of the raw DUO data in the data directory of the project root

TRAIN_JSON = ANNOTATION_DIR / "instances_train_split.json" # set the training JSON path to the instances_train_split.json file within the annotation directory
VAL_JSON = ANNOTATION_DIR / "instances_val_split.json" # set the validation JSON path to the instances_val_split.json file within the annotation directory
TEST_JSON = ANNOTATION_DIR / "instances_test_clean.json" # set the test JSON path to the instances_test_clean.json file within the annotation directory

print("Training images found:", TRAIN_IMAGE_DIR.exists())
print("Test images found:", TEST_IMAGE_DIR.exists())
print("Training JSON found:", TRAIN_JSON.exists())
print("Validation JSON found:", VAL_JSON.exists())
print("Test JSON found:", TEST_JSON.exists())

Training images found: True
Test images found: True
Training JSON found: True
Validation JSON found: True
Test JSON found: True


In [ ]:
from collections import defaultdict # import the defaultdict class from the collections module to create dictionaries with default values for missing keys

import torch
from PIL import Image # import the Image class from the PIL (Python Imaging Library) module for image processing
from torch.utils.data import Dataset # import the Dataset class from the torch.utils.data module to create custom datasets
from torchvision.transforms import functional as F # import the functional module from torchvision.transforms to apply image transformations


class DUOCocoDataset(Dataset): # define a custom dataset class named DUOCocoDataset that inherits from the Dataset class
    def __init__(self, image_dir, annotation_file):
        self.image_dir = Path(image_dir)

        with Path(annotation_file).open("r", encoding="utf-8") as file: # open the annotation JSON file in read mode with UTF-8 encoding
            self.coco = json.load(file)

        self.images = self.coco["images"] # store the list of images from the COCO annotation data in the self.images attribute

        self.annotations_by_image = defaultdict(list) # create a defaultdict to store annotations grouped by image ID, with a default value of an empty list for missing keys

        for annotation in self.coco["annotations"]: # iterate through each annotation in the COCO annotation data
            self.annotations_by_image[
                annotation["image_id"]
            ].append(annotation)

        self.category_ids = sorted( #   create a sorted list of unique category IDs from the COCO annotation data
            category["id"]
            for category in self.coco["categories"]
        )

        # Class 0 is reserved for background
        self.category_to_label = {
            category_id: index + 1
            for index, category_id in enumerate(self.category_ids)
        }

        self.label_to_name = { # create a dictionary mapping category IDs to category names from the COCO annotation data
            self.category_to_label[category["id"]]:
            category["name"]
            for category in self.coco["categories"]
        }

    def __len__(self): # define the __len__ method to return the number of images in the dataset
        return len(self.images)

    def __getitem__(self, index): # define the __getitem__ method to retrieve an image and its corresponding target (annotations) based on the given index
        image_record = self.images[index]

        image_path = ( #    construct the full path to the image file by joining the image directory path with the file name of the image record
            self.image_dir
            / image_record["file_name"]
        )

        image = Image.open(image_path).convert("RGB") # open the image file using PIL and convert it to RGB format
        image = F.to_tensor(image) # convert the PIL image to a PyTorch tensor using torchvision's functional API

        boxes = [] # initialize an empty list 
        labels = []
        areas = []
        iscrowd = []

        annotations = self.annotations_by_image.get( # retrieve the annotations for the current image, defaulting to an empty list if not found
            image_record["id"],
            [],
        )

        for annotation in annotations: # iterate through each annotation associated with the current image
            x, y, width, height = annotation["bbox"]

            x_min = x
            y_min = y
            x_max = x + width
            y_max = y + height

            if x_max <= x_min or y_max <= y_min: # check if the bounding box coordinates are valid (i.e., x_max is greater than x_min and y_max is greater than y_min). If not, skip this annotation and
                continue

            boxes.append( # append the bounding box coordinates to the boxes list in the format [x_min, y_min, x_max, y_max]
                [x_min, y_min, x_max, y_max]
            )

            labels.append( # append the corresponding label for the annotation to the labels list by mapping the category ID to its label using the category_to_label dictionary
                self.category_to_label[
                    annotation["category_id"]
                ]
            )

            areas.append(width * height) # append the area of the bounding box (calculated as width * height) to the areas list
            iscrowd.append(
                annotation.get("iscrowd", 0)
            )

        boxes = torch.as_tensor( # convert the boxes list to a PyTorch tensor of type float32 and reshape it to have a shape of (-1, 4), where -1 indicates that the number of rows will be inferred based on the number of boxes
            boxes,
            dtype=torch.float32,
        ).reshape(-1, 4)

        labels = torch.as_tensor( # convert the labels list to a PyTorch tensor of type int64
            labels,
            dtype=torch.int64,
        )

        areas = torch.as_tensor( # convert the areas list to a PyTorch tensor of type float32
            areas,
            dtype=torch.float32,
        )

        iscrowd = torch.as_tensor( # convert the iscrowd list to a PyTorch tensor of type int64
            iscrowd,
            dtype=torch.int64,
        )

        target = { # create a dictionary representing the target (annotations) for the current image, including the bounding boxes, labels, image ID, areas, and iscrowd flags
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor(
                image_record["id"],
                dtype=torch.int64,
            ),
            "area": areas,
            "iscrowd": iscrowd,
        }

        return image, target # return the image tensor and the target dictionary as a tuple

In [ ]:
#creating the datasets
train_dataset = DUOCocoDataset(
    TRAIN_IMAGE_DIR,
    TRAIN_JSON,
)

val_dataset = DUOCocoDataset( #validation dataset created using the DUOCocoDataset class, with the training image directory and validation JSON file as inputs
    TRAIN_IMAGE_DIR,
    VAL_JSON,
)

test_dataset = DUOCocoDataset( # test dataset created using the DUOCocoDataset class, with the test image directory and test JSON file as inputs
    TEST_IMAGE_DIR,
    TEST_JSON,
)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))
print("Class mapping:", train_dataset.label_to_name)

Training images: 5336
Validation images: 1335
Test images: 1111
Class mapping: {1: 'holothurian', 2: 'echinus', 3: 'scallop', 4: 'starfish'}


In [16]:
#testing one example
image, target = train_dataset[0]

print("Image shape:", image.shape)
print("Boxes shape:", target["boxes"].shape)
print("Labels shape:", target["labels"].shape)
print("First boxes:", target["boxes"][:5])
print("First labels:", target["labels"][:5])

Image shape: torch.Size([3, 405, 720])
Boxes shape: torch.Size([2, 4])
Labels shape: torch.Size([2])
First boxes: tensor([[220., 183., 374., 287.],
        [ 90., 275., 194., 369.]])
First labels: tensor([1, 4])


In [ ]:
#creating and loading faster rnnmodel
from torchvision.models.detection import (
    FasterRCNN_ResNet50_FPN_Weights,
    fasterrcnn_resnet50_fpn,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor #import the FastRCNNPredictor class from the torchvision.models.detection.faster_rcnn module to customize the output layer of the Faster R-CNN model


weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT # load the default weights for the Faster R-CNN model with ResNet50 FPN backbone

model = fasterrcnn_resnet50_fpn( # create a Faster R-CNN model with ResNet50 FPN backbone using the specified weights
    weights=weights,
)

# Four DUO organism classes plus background
num_classes = 5

in_features = ( # get the number of input features for the classifier layer of the Faster R-CNN model by accessing the in_features attribute of the cls_score layer of the box_predictor in the roi_heads of the model
    model.roi_heads
    .box_predictor
    .cls_score
    .in_features
)

model.roi_heads.box_predictor = FastRCNNPredictor( # replace the existing box_predictor of the Faster R-CNN model with a new FastRCNNPredictor that has the specified number of input features and output classes
    in_features,
    num_classes,
)

print("Faster R-CNN loaded successfully.")
print("Number of output classes:", num_classes)

Faster R-CNN loaded successfully.
Number of output classes: 5


In [ ]:
#single batch run
from torch.utils.data import DataLoader


def collate_fn(batch): # define a custom collate function to handle batches of data, which takes a list of tuples (image, target) and returns a tuple of two lists: one for images and one for targets
    return tuple(zip(*batch))


train_loader = DataLoader( # create a DataLoader for the training dataset with a batch size of 1, no shuffling, and the custom collate function
    train_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
)

images, targets = next(iter(train_loader)) # retrieve the first batch of images and targets from the training DataLoader by creating an iterator and using the next() function to get the first item

print("Batch images:", len(images))
print("Batch targets:", len(targets))
print("First image shape:", images[0].shape)
print("First target boxes:", targets[0]["boxes"].shape)

Batch images: 1
Batch targets: 1
First image shape: torch.Size([3, 405, 720])
First target boxes: torch.Size([2, 4])


In [ ]:
#forward pass 
device = torch.device("cpu")
model = model.to(device)
model.train()

images = [ # move images to the specified device
    image.to(device)
    for image in images
]

targets = [ # move targets to the specified device
    {
        key: value.to(device)
        for key, value in target.items()
    }
    for target in targets
]

loss_dict = model(images, targets) # perform a forward pass through the model with the images and targets, which returns a dictionary of loss components

total_loss = sum(loss for loss in loss_dict.values()) # calculate the total loss by summing all the individual loss components from the loss_dict

print("Loss components:")

for name, value in loss_dict.items(): # iterate through each loss component in the loss_dict and print its name and value (detached from the computation graph and converted to a float)
    print(name, float(value.detach()))

print("Total loss:", float(total_loss.detach()))

Loss components:
loss_classifier 1.1216732263565063
loss_box_reg 0.15305393934249878
loss_objectness 0.09559400379657745
loss_rpn_box_reg 0.010313179343938828
Total loss: 1.3806343078613281


In [ ]:
#small ttaining run
import time
from pathlib import Path

import torch
from torch.utils.data import DataLoader, Subset # import the Subset class from the torch.utils.data module to create a subset of the dataset for training


def collate_fn(batch):
    return tuple(zip(*batch)) # define a custom collate function to handle batches of data, which takes a list of tuples (image, target) and returns a tuple of two lists: one for images and one for targets


# Use a very small subset for a pipeline test
small_train_subset = Subset(
    train_dataset,
    range(20),
)

small_train_loader = DataLoader( # create a DataLoader for the small training subset with a batch size of 1, shuffling enabled, and the custom collate function
    small_train_subset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn,
)


device = torch.device("cpu") # set the device to CPU for training. You can change this to "cuda" if you have a compatible GPU available for faster training.
model = model.to(device)

# Train all model parameters
params = [
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad
]

optimizer = torch.optim.SGD( # create an SGD optimizer for the model parameters with the specified learning rate, momentum, and weight decay
    params,
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005,
)


model.train() # set the model to training mode, which enables certain layers (like dropout and batch normalization) to behave differently during training compared to evaluation

start_time = time.time() # record the start time of the training process to measure elapsed time later

epoch_loss = 0.0 # initialis variable to accumulate the total loss for the epoch

for batch_index, (images, targets) in enumerate( # iterate through the small training loader with a starting index of 1
    small_train_loader,
    start=1,
):
    images = [
        image.to(device)
        for image in images
    ]

    targets = [
        {
            key: value.to(device)
            for key, value in target.items()
        }
        for target in targets
    ]

    loss_dict = model(
        images,
        targets,
    )

    total_loss = sum(
        loss
        for loss in loss_dict.values()
    )

    optimizer.zero_grad()
    total_loss.backward()
    optimizer.step()

    batch_loss = float(
        total_loss.detach()
    )

    epoch_loss += batch_loss

    print(
        f"Batch {batch_index:02d}/"
        f"{len(small_train_loader)} "
        f"| Loss: {batch_loss:.4f}"
    )


elapsed_minutes = (
    time.time() - start_time
) / 60

average_loss = (
    epoch_loss
    / len(small_train_loader)
)

print("\nTiny Faster R-CNN test completed.")
print(f"Average loss: {average_loss:.4f}")
print(f"Elapsed time: {elapsed_minutes:.2f} minutes")

Batch 01/20 | Loss: 3.8347
Batch 02/20 | Loss: 1.0644
Batch 03/20 | Loss: 2.3904
Batch 04/20 | Loss: 0.6092
Batch 05/20 | Loss: 0.1659
Batch 06/20 | Loss: 13.2168
Batch 07/20 | Loss: 40.6395
Batch 08/20 | Loss: 44.0821
Batch 09/20 | Loss: 12.8134
Batch 10/20 | Loss: 40392.6875
Batch 11/20 | Loss: nan
Batch 12/20 | Loss: nan
Batch 13/20 | Loss: nan
Batch 14/20 | Loss: nan
Batch 15/20 | Loss: nan
Batch 16/20 | Loss: nan
Batch 17/20 | Loss: nan
Batch 18/20 | Loss: nan
Batch 19/20 | Loss: nan
Batch 20/20 | Loss: nan

Tiny Faster R-CNN test completed.
Average loss: nan
Elapsed time: 3.28 minutes


In [ ]:
from torchvision.models.detection import ( # import the necessary classes for creating a Faster R-CNN model
    FasterRCNN_ResNet50_FPN_Weights,
    fasterrcnn_resnet50_fpn,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor #import the FastRCNNPredictor class from the torchvision.models.detection.faster_rcnn module to customize the output layer of the Faster R-CNN model


device = torch.device("cpu") # set the device to CPU for training.

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT # load the default weights for the Faster R-CNN model with ResNet50 FPN backbone

model = fasterrcnn_resnet50_fpn( # create a Faster R-CNN model with ResNet50 FPN backbone using the specified weights
    weights=weights,
    min_size=416,
    max_size=416,
)

num_classes = 5  # background plus four DUO classes

in_features = ( # get the number of input features for the classifier layer of the Faster R-CNN model by accessing the in_features attribute of the cls_score layer of the box_predictor in the roi_heads of the model
    model.roi_heads
    .box_predictor
    .cls_score
    .in_features
)

model.roi_heads.box_predictor = FastRCNNPredictor( # replace the existing box_predictor of the Faster R-CNN model with a new FastRCNNPredictor that has the specified number of input features and output classes
    in_features,
    num_classes,
)

model = model.to(device) # move the model to the specified device (CPU in this case) for training and inference

print("Fresh Faster R-CNN baseline model created.")
print("Device:", device)
print("Number of classes:", num_classes)
print("Minimum image size:", model.transform.min_size)
print("Maximum image size:", model.transform.max_size)

Fresh Faster R-CNN baseline model created.
Device: cpu
Number of classes: 5
Minimum image size: (416,)
Maximum image size: 416


In [ ]:
#training loader
from torch.utils.data import DataLoader


def collate_fn(batch): # define a custom collate function to handle batches of data, which takes a list of tuples (image, target) and returns a tuple of two lists: one for images and one for targets
    return tuple(zip(*batch))


train_loader = DataLoader( # create a DataLoader for the training dataset with a batch size of 1, shuffling enabled, and the custom collate function
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_fn,
)

val_loader = DataLoader( # create a DataLoader for the validation dataset with a batch size of 1, shuffling disabled, and the custom collate function
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Training batches: 5336
Validation batches: 1335


In [ ]:
trainable_parameters = [ # create a list of parameters that require gradients (i.e., are trainable)
    parameter
    for parameter in model.parameters()
    if parameter.requires_grad
]

optimizer = torch.optim.SGD( # create an SGD optimizer for the trainable parameters of the model with the specified learning rate, momentum, and weight decay
    trainable_parameters,
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005,
)

print("Optimizer created successfully.")
print("Learning rate:", optimizer.param_groups[0]["lr"])

Optimizer created successfully.
Learning rate: 0.005


In [ ]:
#creating run folder and schedular
import csv
import time
from datetime import datetime

RUN_DIR = ( # create a directory path for the current run
    PROJECT_ROOT
    / "results"
    / "faster_rcnn"
    / "duo_faster_rcnn_baseline"
)

RUN_DIR.mkdir(parents=True, exist_ok=True) # create the run directory and any necessary parent directories if they do not already exist

LATEST_CHECKPOINT = RUN_DIR / "latest_checkpoint.pth" # set the path for the latest checkpoint file within the run directory
HISTORY_CSV = RUN_DIR / "training_history.csv"

scheduler = torch.optim.lr_scheduler.StepLR( # create a learning rate scheduler that decays the learning rate of the optimizer by a factor of gamma every step_size epochs
    optimizer,
    step_size=3,
    gamma=0.1,
)

print("Run folder:", RUN_DIR)
print("Checkpoint:", LATEST_CHECKPOINT)
print("History file:", HISTORY_CSV)

Run folder: c:\Users\megdo\Desktop\underwater-object-detection\results\faster_rcnn\duo_faster_rcnn_baseline
Checkpoint: c:\Users\megdo\Desktop\underwater-object-detection\results\faster_rcnn\duo_faster_rcnn_baseline\latest_checkpoint.pth
History file: c:\Users\megdo\Desktop\underwater-object-detection\results\faster_rcnn\duo_faster_rcnn_baseline\training_history.csv


In [ ]:
#full epoch training loop
def train_one_epoch(
    model,
    data_loader,
    optimizer,
    device,
    epoch_number,
):
    model.train()

    epoch_start = time.perf_counter() # record the start time of the epoch to measure elapsed time later
    total_loss = 0.0
    number_of_batches = len(data_loader)

    for batch_number, (images, targets) in enumerate( # iterate through the data_loader with a starting index of 1, retrieving batches of images and targets    
        data_loader,
        start=1,
    ):
        images = [ # move images to the specified device
            image.to(device)
            for image in images
        ]

        targets = [ # move targets to the specified device
            {
                key: value.to(device)
                for key, value in target.items()
            }
            for target in targets
        ]

        loss_dict = model(images, targets) # perform a forward pass through the model with the images and targets, which returns a dictionary of loss components

        losses = sum(
            loss
            for loss in loss_dict.values()
        )

        if not torch.isfinite(losses): # check if the computed losses are finite (i.e., not NaN or infinite). If not, raise a RuntimeError with the loss value
            raise RuntimeError(
                f"Non-finite loss encountered: {losses.item()}"
            )

        optimizer.zero_grad() # clear the gradients of all optimise parameters to prepare for the backward pass
        losses.backward()
        optimizer.step()

        batch_loss = float(losses.detach()) # compute the batch loss by detaching the losses tensor from the computation graph and converting it to a float
        total_loss += batch_loss

        if batch_number == 1 or batch_number % 100 == 0:
            elapsed_hours = (
                time.perf_counter() - epoch_start
            ) / 3600

            average_so_far = total_loss / batch_number

            print(
                f"Epoch {epoch_number} | "
                f"Batch {batch_number}/{number_of_batches} | "
                f"Batch loss: {batch_loss:.4f} | "
                f"Average loss: {average_so_far:.4f} | "
                f"Elapsed: {elapsed_hours:.2f} hours"
            )

    epoch_seconds = time.perf_counter() - epoch_start
    average_loss = total_loss / number_of_batches

    return average_loss, epoch_seconds # return the average loss and elapsed time for the epoch in seconds

In [ ]:
#checkpoints and history saving
def append_history(
    epoch_number,
    average_train_loss,
    epoch_seconds,
    learning_rate,
):
    file_exists = HISTORY_CSV.exists() # check if the history CSV file already exists to determine whether to write the header row or not

    with HISTORY_CSV.open( # open the history CSV file in append mode
        "a",
        newline="",
        encoding="utf-8",
    ) as file:
        writer = csv.writer(file)

        if not file_exists:
            writer.writerow( # write the header row to the CSV file if it does not already exist
                [
                    "epoch",
                    "average_train_loss",
                    "epoch_seconds",
                    "epoch_hours",
                    "learning_rate",
                    "completion_time",
                ]
            )

        writer.writerow( # write the training history for the current epoch
            [
                epoch_number,
                average_train_loss,
                epoch_seconds,
                epoch_seconds / 3600,
                learning_rate,
                datetime.now().isoformat(
                    timespec="seconds"
                ),
            ]
        )


def save_checkpoint( # save a checkpoint of the model and optimizer state
    epoch_number,
    average_train_loss,
    cumulative_training_seconds,
):
    torch.save(
        {
            "completed_epoch": epoch_number, # save the number of completed epochs in the checkpoint
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict":
                optimizer.state_dict(),
            "scheduler_state_dict":
                scheduler.state_dict(),
            "average_train_loss":
                average_train_loss,
            "cumulative_training_seconds":
                cumulative_training_seconds,
        },
        LATEST_CHECKPOINT,
    )

    epoch_checkpoint = ( # create a path for the epoch-specific checkpoint file within the
        RUN_DIR
        / f"checkpoint_epoch_{epoch_number:02d}.pth"
    )

    torch.save(
        {
            "completed_epoch": epoch_number,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict":
                optimizer.state_dict(),
            "scheduler_state_dict":
                scheduler.state_dict(),
            "average_train_loss":
                average_train_loss,
            "cumulative_training_seconds":
                cumulative_training_seconds,
        },
        epoch_checkpoint,
    )

    print("Latest checkpoint:", LATEST_CHECKPOINT)
    print("Epoch checkpoint:", epoch_checkpoint)

In [ ]:
#starting epoch 1 training loop
epoch_number = 1
cumulative_training_seconds = 0.0 # initialize a variable to keep track of the cumulative training time across epochs

current_learning_rate = ( # retrieve the current learning rate from the optimizer's parameter groups, specifically from the first parameter group
    optimizer.param_groups[0]["lr"]
)

print("Beginning full Faster R-CNN training")
print("Epoch:", epoch_number)
print("Training images:", len(train_dataset))
print("Learning rate:", current_learning_rate)
print("Start time:", datetime.now())

average_train_loss, epoch_seconds = train_one_epoch( #apply the train_one_epoch function to train the model for one epoch, passing in the model, training data loader, optimizer, device, and epoch number as arguments. The function returns the average training loss and the time taken for the epoch in seconds
    model=model,
    data_loader=train_loader,
    optimizer=optimizer,
    device=device,
    epoch_number=epoch_number,
)

cumulative_training_seconds += epoch_seconds

# Update the learning rate after completing the epoch
scheduler.step()

append_history( # append the training history for the current epoch to the CSV file
    epoch_number=epoch_number,
    average_train_loss=average_train_loss,
    epoch_seconds=epoch_seconds,
    learning_rate=current_learning_rate,
)

save_checkpoint( # save a checkpoint of the model and optimizer state after completing the epoch
    epoch_number=epoch_number,
    average_train_loss=average_train_loss,
    cumulative_training_seconds=
        cumulative_training_seconds,
)

print("\nEpoch completed successfully.")
print(f"Average loss: {average_train_loss:.4f}")
print(f"Epoch time: {epoch_seconds / 3600:.2f} hours")
print(
    "Cumulative training time:",
    f"{cumulative_training_seconds / 3600:.2f} hours",
)
print("Finish time:", datetime.now())

Beginning full Faster R-CNN training
Epoch: 1
Training images: 5336
Learning rate: 0.005
Start time: 2026-07-18 12:07:30.807027
Epoch 1 | Batch 1/5336 | Batch loss: 4.9245 | Average loss: 4.9245 | Elapsed: 0.00 hours
Epoch 1 | Batch 100/5336 | Batch loss: 0.7844 | Average loss: 1.1257 | Elapsed: 0.04 hours
Epoch 1 | Batch 200/5336 | Batch loss: 0.5880 | Average loss: 1.0430 | Elapsed: 0.07 hours
Epoch 1 | Batch 300/5336 | Batch loss: 0.6649 | Average loss: 0.9828 | Elapsed: 0.11 hours
Epoch 1 | Batch 400/5336 | Batch loss: 2.3761 | Average loss: 0.9514 | Elapsed: 0.15 hours
Epoch 1 | Batch 500/5336 | Batch loss: 0.5604 | Average loss: 0.9262 | Elapsed: 0.18 hours
Epoch 1 | Batch 600/5336 | Batch loss: 1.0142 | Average loss: 0.8984 | Elapsed: 0.22 hours
Epoch 1 | Batch 700/5336 | Batch loss: 0.4593 | Average loss: 0.8833 | Elapsed: 0.26 hours
Epoch 1 | Batch 800/5336 | Batch loss: 0.5595 | Average loss: 0.8608 | Elapsed: 0.30 hours
Epoch 1 | Batch 900/5336 | Batch loss: 1.3337 | Average

In [23]:
#checking if the latest checkpoint exists and printing its path
print(LATEST_CHECKPOINT.exists())
print(LATEST_CHECKPOINT)

True
c:\Users\megdo\Desktop\underwater-object-detection\results\faster_rcnn\duo_faster_rcnn_baseline\latest_checkpoint.pth


In [ ]:
#load the latest checkpoint and resume training
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict( # load the model state from the checkpoint
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict( # load the optimizer state from the checkpoint
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict( # load the scheduler state from the checkpoint
    checkpoint["scheduler_state_dict"]
)

completed_epoch = checkpoint["completed_epoch"] # retrieve the number of completed epochs from the checkpoint

cumulative_training_seconds = checkpoint[
    "cumulative_training_seconds"
]

epoch_number = completed_epoch + 1 # set the next epoch number to be one greater than the completed epoch

print("Completed epoch:", completed_epoch)
print("Next epoch:", epoch_number)
print(
    "Cumulative training time:",
    f"{cumulative_training_seconds / 3600:.2f} hours",
)
print(
    "Current learning rate:",
    optimizer.param_groups[0]["lr"],
)

Completed epoch: 1
Next epoch: 2
Cumulative training time: 2.01 hours
Current learning rate: 0.005


In [ ]:
current_learning_rate = optimizer.param_groups[0]["lr"] # retrieve the current learning rate from the optimizer's parameter groups, specifically from the first parameter group

In [ ]:
#full training epoch loop
print("Beginning Faster R-CNN training")
print("Epoch:", epoch_number)
print("Training images:", len(train_dataset))
print("Learning rate:", current_learning_rate)
print("Start time:", datetime.now())

average_train_loss, epoch_seconds = train_one_epoch( #apply the train_one_epoch function to train the model for one epoch, passing in the model, training data loader, optimizer, device, and epoch number as arguments. The function returns the average training loss and the time taken for the epoch in seconds
    model=model,
    data_loader=train_loader,
    optimizer=optimizer,
    device=device,
    epoch_number=epoch_number,
)

cumulative_training_seconds += epoch_seconds # update the cumulative training time by adding the time taken for the current epoch

scheduler.step() # update the learning rate using the scheduler after completing the epoch

append_history( # append the training history for the current epoch to the CSV file
    epoch_number=epoch_number,
    average_train_loss=average_train_loss,
    epoch_seconds=epoch_seconds,
    learning_rate=current_learning_rate,
)

save_checkpoint( # save a checkpoint of the model and optimizer state after completing the epoch
    epoch_number=epoch_number,
    average_train_loss=average_train_loss,
    cumulative_training_seconds=cumulative_training_seconds,
)

print("\nEpoch completed successfully.")
print(f"Average loss: {average_train_loss:.4f}")
print(f"Epoch time: {epoch_seconds / 3600:.2f} hours")
print(
    "Cumulative training time:",
    f"{cumulative_training_seconds / 3600:.2f} hours",
)
print("Finish time:", datetime.now())

Beginning Faster R-CNN training
Epoch: 2
Training images: 5336
Learning rate: 0.005
Start time: 2026-07-18 14:33:46.058296
Epoch 2 | Batch 1/5336 | Batch loss: 0.4862 | Average loss: 0.4862 | Elapsed: 0.00 hours
Epoch 2 | Batch 100/5336 | Batch loss: 0.4615 | Average loss: 0.5111 | Elapsed: 0.04 hours
Epoch 2 | Batch 200/5336 | Batch loss: 0.3099 | Average loss: 0.4905 | Elapsed: 0.08 hours
Epoch 2 | Batch 300/5336 | Batch loss: 0.8913 | Average loss: 0.4976 | Elapsed: 0.11 hours
Epoch 2 | Batch 400/5336 | Batch loss: 0.1478 | Average loss: 0.4916 | Elapsed: 0.15 hours
Epoch 2 | Batch 500/5336 | Batch loss: 0.2796 | Average loss: 0.4964 | Elapsed: 0.19 hours
Epoch 2 | Batch 600/5336 | Batch loss: 0.4941 | Average loss: 0.4912 | Elapsed: 0.23 hours
Epoch 2 | Batch 700/5336 | Batch loss: 0.7342 | Average loss: 0.4861 | Elapsed: 0.26 hours
Epoch 2 | Batch 800/5336 | Batch loss: 0.5623 | Average loss: 0.4896 | Elapsed: 0.30 hours
Epoch 2 | Batch 900/5336 | Batch loss: 0.4165 | Average loss

In [27]:
#preparing epoch 3
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

completed_epoch = checkpoint["completed_epoch"]

cumulative_training_seconds = checkpoint[
    "cumulative_training_seconds"
]

epoch_number = completed_epoch + 1

current_learning_rate = optimizer.param_groups[0]["lr"]

print("Completed epoch:", completed_epoch)
print("Next epoch:", epoch_number)
print(
    "Cumulative training time:",
    f"{cumulative_training_seconds / 3600:.2f} hours",
)
print("Current learning rate:", current_learning_rate)

Completed epoch: 2
Next epoch: 3
Cumulative training time: 4.08 hours
Current learning rate: 0.005


In [28]:
#starting epoch 3 training 
print("Beginning Faster R-CNN training")
print("Epoch:", epoch_number)
print("Training images:", len(train_dataset))
print("Learning rate:", current_learning_rate)
print("Start time:", datetime.now())

average_train_loss, epoch_seconds = train_one_epoch(
    model=model,
    data_loader=train_loader,
    optimizer=optimizer,
    device=device,
    epoch_number=epoch_number,
)

cumulative_training_seconds += epoch_seconds

scheduler.step()

append_history(
    epoch_number=epoch_number,
    average_train_loss=average_train_loss,
    epoch_seconds=epoch_seconds,
    learning_rate=current_learning_rate,
)

save_checkpoint(
    epoch_number=epoch_number,
    average_train_loss=average_train_loss,
    cumulative_training_seconds=cumulative_training_seconds,
)

print("\nEpoch completed successfully.")
print(f"Average loss: {average_train_loss:.4f}")
print(f"Epoch time: {epoch_seconds / 3600:.2f} hours")
print(
    "Cumulative training time:",
    f"{cumulative_training_seconds / 3600:.2f} hours",
)
print("Finish time:", datetime.now())

Beginning Faster R-CNN training
Epoch: 3
Training images: 5336
Learning rate: 0.005
Start time: 2026-07-18 17:12:37.657865
Epoch 3 | Batch 1/5336 | Batch loss: 0.5626 | Average loss: 0.5626 | Elapsed: 0.00 hours
Epoch 3 | Batch 100/5336 | Batch loss: 0.8738 | Average loss: 0.4630 | Elapsed: 0.07 hours
Epoch 3 | Batch 200/5336 | Batch loss: 0.2905 | Average loss: 0.4454 | Elapsed: 0.13 hours
Epoch 3 | Batch 300/5336 | Batch loss: 0.4342 | Average loss: 0.4576 | Elapsed: 0.20 hours
Epoch 3 | Batch 400/5336 | Batch loss: 0.3768 | Average loss: 0.4570 | Elapsed: 0.27 hours
Epoch 3 | Batch 500/5336 | Batch loss: 0.1633 | Average loss: 0.4482 | Elapsed: 0.34 hours
Epoch 3 | Batch 600/5336 | Batch loss: 0.3442 | Average loss: 0.4419 | Elapsed: 0.41 hours
Epoch 3 | Batch 700/5336 | Batch loss: 0.3275 | Average loss: 0.4396 | Elapsed: 0.48 hours
Epoch 3 | Batch 800/5336 | Batch loss: 0.5660 | Average loss: 0.4411 | Elapsed: 0.54 hours
Epoch 3 | Batch 900/5336 | Batch loss: 0.3742 | Average loss

In [29]:
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

completed_epoch = checkpoint["completed_epoch"]

cumulative_training_seconds = checkpoint[
    "cumulative_training_seconds"
]

epoch_number = completed_epoch + 1

current_learning_rate = optimizer.param_groups[0]["lr"]

print("Completed epoch:", completed_epoch)
print("Next epoch:", epoch_number)
print(
    "Cumulative training time:",
    f"{cumulative_training_seconds / 3600:.2f} hours",
)
print("Current learning rate:", current_learning_rate)

Completed epoch: 3
Next epoch: 4
Cumulative training time: 6.67 hours
Current learning rate: 0.0005


In [30]:
#running epoch 4
print("Beginning Faster R-CNN training")
print("Epoch:", epoch_number)
print("Training images:", len(train_dataset))
print("Learning rate:", current_learning_rate)
print("Start time:", datetime.now())

average_train_loss, epoch_seconds = train_one_epoch(
    model=model,
    data_loader=train_loader,
    optimizer=optimizer,
    device=device,
    epoch_number=epoch_number,
)

cumulative_training_seconds += epoch_seconds

scheduler.step()

append_history(
    epoch_number=epoch_number,
    average_train_loss=average_train_loss,
    epoch_seconds=epoch_seconds,
    learning_rate=current_learning_rate,
)

save_checkpoint(
    epoch_number=epoch_number,
    average_train_loss=average_train_loss,
    cumulative_training_seconds=cumulative_training_seconds,
)

print("\nEpoch completed successfully.")
print(f"Average loss: {average_train_loss:.4f}")
print(f"Epoch time: {epoch_seconds / 3600:.2f} hours")
print(
    "Cumulative training time:",
    f"{cumulative_training_seconds / 3600:.2f} hours",
)
print("Finish time:", datetime.now())

Beginning Faster R-CNN training
Epoch: 4
Training images: 5336
Learning rate: 0.0005
Start time: 2026-07-18 20:13:53.787466
Epoch 4 | Batch 1/5336 | Batch loss: 0.2279 | Average loss: 0.2279 | Elapsed: 0.00 hours
Epoch 4 | Batch 100/5336 | Batch loss: 0.2633 | Average loss: 0.3871 | Elapsed: 0.06 hours
Epoch 4 | Batch 200/5336 | Batch loss: 0.0569 | Average loss: 0.3717 | Elapsed: 0.10 hours
Epoch 4 | Batch 300/5336 | Batch loss: 0.0909 | Average loss: 0.3762 | Elapsed: 0.13 hours
Epoch 4 | Batch 400/5336 | Batch loss: 0.1477 | Average loss: 0.3665 | Elapsed: 0.17 hours
Epoch 4 | Batch 500/5336 | Batch loss: 0.1772 | Average loss: 0.3731 | Elapsed: 0.21 hours
Epoch 4 | Batch 600/5336 | Batch loss: 0.2462 | Average loss: 0.3688 | Elapsed: 0.25 hours
Epoch 4 | Batch 700/5336 | Batch loss: 0.1454 | Average loss: 0.3618 | Elapsed: 0.29 hours
Epoch 4 | Batch 800/5336 | Batch loss: 0.2661 | Average loss: 0.3600 | Elapsed: 0.32 hours
Epoch 4 | Batch 900/5336 | Batch loss: 0.1301 | Average los

In [31]:
#new epoch
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

completed_epoch = checkpoint["completed_epoch"]

cumulative_training_seconds = checkpoint[
    "cumulative_training_seconds"
]

epoch_number = completed_epoch + 1

current_learning_rate = optimizer.param_groups[0]["lr"]

print("Completed epoch:", completed_epoch)
print("Next epoch:", epoch_number)
print(
    "Cumulative training time:",
    f"{cumulative_training_seconds / 3600:.2f} hours",
)
print("Current learning rate:", current_learning_rate)

Completed epoch: 4
Next epoch: 5
Cumulative training time: 8.70 hours
Current learning rate: 0.0005


In [32]:
#epoch 5 training
print("Beginning Faster R-CNN training")
print("Epoch:", epoch_number)
print("Training images:", len(train_dataset))
print("Learning rate:", current_learning_rate)
print("Start time:", datetime.now())

average_train_loss, epoch_seconds = train_one_epoch(
    model=model,
    data_loader=train_loader,
    optimizer=optimizer,
    device=device,
    epoch_number=epoch_number,
)

cumulative_training_seconds += epoch_seconds

scheduler.step()

append_history(
    epoch_number=epoch_number,
    average_train_loss=average_train_loss,
    epoch_seconds=epoch_seconds,
    learning_rate=current_learning_rate,
)

save_checkpoint(
    epoch_number=epoch_number,
    average_train_loss=average_train_loss,
    cumulative_training_seconds=cumulative_training_seconds,
)

print("\nEpoch completed successfully.")
print(f"Average loss: {average_train_loss:.4f}")
print(f"Epoch time: {epoch_seconds / 3600:.2f} hours")
print(
    "Cumulative training time:",
    f"{cumulative_training_seconds / 3600:.2f} hours",
)
print("Finish time:", datetime.now())

Beginning Faster R-CNN training
Epoch: 5
Training images: 5336
Learning rate: 0.0005
Start time: 2026-07-19 18:38:16.821629
Epoch 5 | Batch 1/5336 | Batch loss: 0.2057 | Average loss: 0.2057 | Elapsed: 0.00 hours
Epoch 5 | Batch 100/5336 | Batch loss: 0.2505 | Average loss: 0.3436 | Elapsed: 0.05 hours
Epoch 5 | Batch 200/5336 | Batch loss: 0.6308 | Average loss: 0.3431 | Elapsed: 0.12 hours
Epoch 5 | Batch 300/5336 | Batch loss: 0.2118 | Average loss: 0.3332 | Elapsed: 0.59 hours
Epoch 5 | Batch 400/5336 | Batch loss: 0.0143 | Average loss: 0.3212 | Elapsed: 0.64 hours
Epoch 5 | Batch 500/5336 | Batch loss: 0.0531 | Average loss: 0.3156 | Elapsed: 0.71 hours
Epoch 5 | Batch 600/5336 | Batch loss: 0.0145 | Average loss: 0.3177 | Elapsed: 1.24 hours
Epoch 5 | Batch 700/5336 | Batch loss: 0.4226 | Average loss: 0.3215 | Elapsed: 16.47 hours
Epoch 5 | Batch 800/5336 | Batch loss: 0.4579 | Average loss: 0.3153 | Elapsed: 16.53 hours
Epoch 5 | Batch 900/5336 | Batch loss: 0.4291 | Average l

In [33]:
#checking checkpoints
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

print("Completed epoch:", checkpoint["completed_epoch"])
print(
    "Saved cumulative time:",
    checkpoint["cumulative_training_seconds"] / 3600,
    "hours",
)

Completed epoch: 5
Saved cumulative time: 28.980617196972222 hours


In [34]:
def calculate_validation_loss(
    model,
    data_loader,
    device,
):
    model.train()

    validation_start = time.perf_counter()
    total_validation_loss = 0.0
    number_of_batches = len(data_loader)

    with torch.no_grad():
        for batch_number, (images, targets) in enumerate(
            data_loader,
            start=1,
        ):
            images = [
                image.to(device)
                for image in images
            ]

            targets = [
                {
                    key: value.to(device)
                    for key, value in target.items()
                }
                for target in targets
            ]

            loss_dict = model(images, targets)

            total_loss = sum(
                loss
                for loss in loss_dict.values()
            )

            total_validation_loss += float(total_loss)

            if batch_number == 1 or batch_number % 100 == 0:
                print(
                    f"Validation batch "
                    f"{batch_number}/{number_of_batches}"
                )

    average_validation_loss = (
        total_validation_loss / number_of_batches
    )

    validation_seconds = (
        time.perf_counter() - validation_start
    )

    return average_validation_loss, validation_seconds

In [35]:
average_validation_loss, validation_seconds = (
    calculate_validation_loss(
        model=model,
        data_loader=val_loader,
        device=device,
    )
)

print(
    "Average validation loss:",
    f"{average_validation_loss:.4f}",
)

print(
    "Validation time:",
    f"{validation_seconds / 3600:.2f} hours",
)

Validation batch 1/1335
Validation batch 100/1335
Validation batch 200/1335
Validation batch 300/1335
Validation batch 400/1335
Validation batch 500/1335
Validation batch 600/1335
Validation batch 700/1335
Validation batch 800/1335
Validation batch 900/1335
Validation batch 1000/1335
Validation batch 1100/1335
Validation batch 1200/1335
Validation batch 1300/1335
Average validation loss: 0.3752
Validation time: 0.16 hours


In [36]:
#epoch checkpoint
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

completed_epoch = checkpoint["completed_epoch"]

cumulative_training_seconds = checkpoint[
    "cumulative_training_seconds"
]

epoch_number = completed_epoch + 1

current_learning_rate = optimizer.param_groups[0]["lr"]

print("Completed epoch:", completed_epoch)
print("Next epoch:", epoch_number)
print(
    "Cumulative training time:",
    f"{cumulative_training_seconds / 3600:.2f} hours",
)
print("Current learning rate:", current_learning_rate)

Completed epoch: 5
Next epoch: 6
Cumulative training time: 28.98 hours
Current learning rate: 0.0005


In [37]:
#running epoch 6
print("Beginning Faster R-CNN training")
print("Epoch:", epoch_number)
print("Training images:", len(train_dataset))
print("Learning rate:", current_learning_rate)
print("Start time:", datetime.now())

average_train_loss, epoch_seconds = train_one_epoch(
    model=model,
    data_loader=train_loader,
    optimizer=optimizer,
    device=device,
    epoch_number=epoch_number,
)

cumulative_training_seconds += epoch_seconds

scheduler.step()

append_history(
    epoch_number=epoch_number,
    average_train_loss=average_train_loss,
    epoch_seconds=epoch_seconds,
    learning_rate=current_learning_rate,
)

save_checkpoint(
    epoch_number=epoch_number,
    average_train_loss=average_train_loss,
    cumulative_training_seconds=cumulative_training_seconds,
)

print("\nEpoch completed successfully.")
print(f"Average loss: {average_train_loss:.4f}")
print(f"Epoch time: {epoch_seconds / 3600:.2f} hours")
print(
    "Cumulative training time:",
    f"{cumulative_training_seconds / 3600:.2f} hours",
)
print("Finish time:", datetime.now())

Beginning Faster R-CNN training
Epoch: 6
Training images: 5336
Learning rate: 0.0005
Start time: 2026-07-20 15:33:18.302481
Epoch 6 | Batch 1/5336 | Batch loss: 0.3461 | Average loss: 0.3461 | Elapsed: 0.00 hours
Epoch 6 | Batch 100/5336 | Batch loss: 0.0587 | Average loss: 0.3072 | Elapsed: 0.07 hours
Epoch 6 | Batch 200/5336 | Batch loss: 0.1534 | Average loss: 0.3140 | Elapsed: 0.13 hours
Epoch 6 | Batch 300/5336 | Batch loss: 0.1294 | Average loss: 0.3201 | Elapsed: 0.20 hours
Epoch 6 | Batch 400/5336 | Batch loss: 0.8777 | Average loss: 0.3191 | Elapsed: 0.27 hours
Epoch 6 | Batch 500/5336 | Batch loss: 0.3007 | Average loss: 0.3108 | Elapsed: 0.33 hours
Epoch 6 | Batch 600/5336 | Batch loss: 0.7689 | Average loss: 0.3097 | Elapsed: 0.40 hours
Epoch 6 | Batch 700/5336 | Batch loss: 0.0281 | Average loss: 0.3031 | Elapsed: 0.47 hours
Epoch 6 | Batch 800/5336 | Batch loss: 0.7104 | Average loss: 0.3000 | Elapsed: 0.54 hours
Epoch 6 | Batch 900/5336 | Batch loss: 0.6751 | Average los

In [38]:
#validatin epoch 6 checkpoint
average_validation_loss, validation_seconds = (
    calculate_validation_loss(
        model=model,
        data_loader=val_loader,
        device=device,
    )
)

print(
    "Average validation loss:",
    f"{average_validation_loss:.4f}",
)

print(
    "Validation time:",
    f"{validation_seconds / 3600:.2f} hours",
)

Validation batch 1/1335
Validation batch 100/1335
Validation batch 200/1335
Validation batch 300/1335
Validation batch 400/1335
Validation batch 500/1335
Validation batch 600/1335
Validation batch 700/1335
Validation batch 800/1335
Validation batch 900/1335
Validation batch 1000/1335
Validation batch 1100/1335
Validation batch 1200/1335
Validation batch 1300/1335
Average validation loss: 0.3730
Validation time: 0.16 hours


In [39]:
#preparing epoch 7
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

completed_epoch = checkpoint["completed_epoch"]

cumulative_training_seconds = checkpoint[
    "cumulative_training_seconds"
]

epoch_number = completed_epoch + 1

current_learning_rate = optimizer.param_groups[0]["lr"]

print("Completed epoch:", completed_epoch)
print("Next epoch:", epoch_number)
print(
    "Cumulative training time:",
    f"{cumulative_training_seconds / 3600:.2f} hours",
)
print("Current learning rate:", current_learning_rate)

Completed epoch: 6
Next epoch: 7
Cumulative training time: 31.54 hours
Current learning rate: 5e-05


In [40]:
#starting epoch 7
print("Beginning Faster R-CNN training")
print("Epoch:", epoch_number)
print("Training images:", len(train_dataset))
print("Learning rate:", current_learning_rate)
print("Start time:", datetime.now())

average_train_loss, epoch_seconds = train_one_epoch(
    model=model,
    data_loader=train_loader,
    optimizer=optimizer,
    device=device,
    epoch_number=epoch_number,
)

cumulative_training_seconds += epoch_seconds

scheduler.step()

append_history(
    epoch_number=epoch_number,
    average_train_loss=average_train_loss,
    epoch_seconds=epoch_seconds,
    learning_rate=current_learning_rate,
)

save_checkpoint(
    epoch_number=epoch_number,
    average_train_loss=average_train_loss,
    cumulative_training_seconds=cumulative_training_seconds,
)

print("\nEpoch completed successfully.")
print(f"Average loss: {average_train_loss:.4f}")
print(f"Epoch time: {epoch_seconds / 3600:.2f} hours")
print(
    "Cumulative training time:",
    f"{cumulative_training_seconds / 3600:.2f} hours",
)
print("Finish time:", datetime.now())

Beginning Faster R-CNN training
Epoch: 7
Training images: 5336
Learning rate: 5e-05
Start time: 2026-07-20 19:26:18.576771
Epoch 7 | Batch 1/5336 | Batch loss: 0.5298 | Average loss: 0.5298 | Elapsed: 0.00 hours
Epoch 7 | Batch 100/5336 | Batch loss: 0.1784 | Average loss: 0.2512 | Elapsed: 0.04 hours
Epoch 7 | Batch 200/5336 | Batch loss: 0.0531 | Average loss: 0.2699 | Elapsed: 0.07 hours
Epoch 7 | Batch 300/5336 | Batch loss: 0.3971 | Average loss: 0.2766 | Elapsed: 0.11 hours
Epoch 7 | Batch 400/5336 | Batch loss: 0.7913 | Average loss: 0.2790 | Elapsed: 0.15 hours
Epoch 7 | Batch 500/5336 | Batch loss: 0.3233 | Average loss: 0.2790 | Elapsed: 0.18 hours
Epoch 7 | Batch 600/5336 | Batch loss: 0.1681 | Average loss: 0.2787 | Elapsed: 0.22 hours
Epoch 7 | Batch 700/5336 | Batch loss: 0.1922 | Average loss: 0.2765 | Elapsed: 0.26 hours
Epoch 7 | Batch 800/5336 | Batch loss: 0.4392 | Average loss: 0.2817 | Elapsed: 0.30 hours
Epoch 7 | Batch 900/5336 | Batch loss: 0.2519 | Average loss

In [ ]:
average_validation_loss, validation_seconds = (
    calculate_validation_loss(
        model=model,
        data_loader=val_loader,
        device=device,
    )
)

print(
    "Average validation loss:",
    f"{average_validation_loss:.4f}",
)

print(
    "Validation time:",
    f"{validation_seconds / 3600:.2f} hours",
)

Validation batch 1/1335
Validation batch 100/1335
Validation batch 200/1335
Validation batch 300/1335
Validation batch 400/1335
Validation batch 500/1335
Validation batch 600/1335
Validation batch 700/1335
Validation batch 800/1335
Validation batch 900/1335
Validation batch 1000/1335
Validation batch 1100/1335
Validation batch 1200/1335
Validation batch 1300/1335
Average validation loss: 0.3802
Validation time: 0.22 hours


In [30]:
#preparing epoch 8
checkpoint = torch.load(
    LATEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

completed_epoch = checkpoint["completed_epoch"]

cumulative_training_seconds = checkpoint[
    "cumulative_training_seconds"
]

epoch_number = completed_epoch + 1

current_learning_rate = optimizer.param_groups[0]["lr"]

print("Completed epoch:", completed_epoch)
print("Next epoch:", epoch_number)
print(
    "Cumulative training time:",
    f"{cumulative_training_seconds / 3600:.2f} hours",
)
print("Current learning rate:", current_learning_rate)

Completed epoch: 7
Next epoch: 8
Cumulative training time: 33.56 hours
Current learning rate: 5e-05


In [29]:
import torch

print(torch.__version__)

2.13.0+cpu


In [31]:
print("Beginning Faster R-CNN training")
print("Epoch:", epoch_number)
print("Training images:", len(train_dataset))
print("Learning rate:", current_learning_rate)
print("Start time:", datetime.now())

average_train_loss, epoch_seconds = train_one_epoch(
    model=model,
    data_loader=train_loader,
    optimizer=optimizer,
    device=device,
    epoch_number=epoch_number,
)

cumulative_training_seconds += epoch_seconds

scheduler.step()

append_history(
    epoch_number=epoch_number,
    average_train_loss=average_train_loss,
    epoch_seconds=epoch_seconds,
    learning_rate=current_learning_rate,
)

save_checkpoint(
    epoch_number=epoch_number,
    average_train_loss=average_train_loss,
    cumulative_training_seconds=cumulative_training_seconds,
)

print("\nEpoch completed successfully.")
print(f"Average loss: {average_train_loss:.4f}")
print(f"Epoch time: {epoch_seconds / 3600:.2f} hours")
print(
    "Cumulative training time:",
    f"{cumulative_training_seconds / 3600:.2f} hours",
)
print("Finish time:", datetime.now())

Beginning Faster R-CNN training
Epoch: 8
Training images: 5336
Learning rate: 5e-05
Start time: 2026-07-21 11:41:03.120765
Epoch 8 | Batch 1/5336 | Batch loss: 0.1219 | Average loss: 0.1219 | Elapsed: 0.00 hours
Epoch 8 | Batch 100/5336 | Batch loss: 0.5268 | Average loss: 0.2780 | Elapsed: 0.04 hours
Epoch 8 | Batch 200/5336 | Batch loss: 0.1702 | Average loss: 0.2829 | Elapsed: 0.08 hours
Epoch 8 | Batch 300/5336 | Batch loss: 0.2426 | Average loss: 0.2804 | Elapsed: 0.12 hours
Epoch 8 | Batch 400/5336 | Batch loss: 0.2415 | Average loss: 0.2761 | Elapsed: 0.15 hours
Epoch 8 | Batch 500/5336 | Batch loss: 0.1215 | Average loss: 0.2781 | Elapsed: 0.19 hours
Epoch 8 | Batch 600/5336 | Batch loss: 0.0631 | Average loss: 0.2767 | Elapsed: 0.23 hours
Epoch 8 | Batch 700/5336 | Batch loss: 0.2686 | Average loss: 0.2735 | Elapsed: 0.27 hours
Epoch 8 | Batch 800/5336 | Batch loss: 0.2392 | Average loss: 0.2778 | Elapsed: 0.30 hours
Epoch 8 | Batch 900/5336 | Batch loss: 0.0420 | Average loss

In [33]:
def calculate_validation_loss(
    model,
    data_loader,
    device,
):
    model.train()

    validation_start = time.perf_counter()
    total_validation_loss = 0.0
    number_of_batches = len(data_loader)

    with torch.no_grad():
        for batch_number, (images, targets) in enumerate(
            data_loader,
            start=1,
        ):
            images = [
                image.to(device)
                for image in images
            ]

            targets = [
                {
                    key: value.to(device)
                    for key, value in target.items()
                }
                for target in targets
            ]

            loss_dict = model(images, targets)

            total_loss = sum(
                loss
                for loss in loss_dict.values()
            )

            total_validation_loss += float(total_loss)

            if batch_number == 1 or batch_number % 100 == 0:
                print(
                    f"Validation batch "
                    f"{batch_number}/{number_of_batches}"
                )

    average_validation_loss = (
        total_validation_loss / number_of_batches
    )

    validation_seconds = (
        time.perf_counter() - validation_start
    )

    return average_validation_loss, validation_seconds

In [34]:
average_validation_loss, validation_seconds = (
    calculate_validation_loss(
        model=model,
        data_loader=val_loader,
        device=device,
    )
)

print(
    "Average validation loss:",
    f"{average_validation_loss:.4f}",
)

print(
    "Validation time:",
    f"{validation_seconds / 3600:.2f} hours",
)

Validation batch 1/1335
Validation batch 100/1335
Validation batch 200/1335
Validation batch 300/1335
Validation batch 400/1335
Validation batch 500/1335
Validation batch 600/1335
Validation batch 700/1335
Validation batch 800/1335
Validation batch 900/1335
Validation batch 1000/1335
Validation batch 1100/1335
Validation batch 1200/1335
Validation batch 1300/1335
Average validation loss: 0.3835
Validation time: 0.25 hours


In [ ]:
BEST_CHECKPOINT = ( # define the path for the best checkpoint file
    RUN_DIR
    / "checkpoint_epoch_05.pth"
)

print("Checkpoint exists:", BEST_CHECKPOINT.exists())
print("Checkpoint path:", BEST_CHECKPOINT)

Checkpoint exists: True
Checkpoint path: c:\Users\megdo\Desktop\underwater-object-detection\results\faster_rcnn\duo_faster_rcnn_baseline\checkpoint_epoch_05.pth


In [ ]:
best_checkpoint = torch.load( # load the best checkpoint from the specified path
    BEST_CHECKPOINT,
    map_location=device,
)

model.load_state_dict( # load the model state from the best checkpoint
    best_checkpoint["model_state_dict"]
)

model = model.to(device) 

print(
    "Loaded checkpoint from epoch:",
    best_checkpoint["completed_epoch"],
)

print(
    "Training loss at saved epoch:",
    best_checkpoint["average_train_loss"],
)

Loaded checkpoint from epoch: 5
Training loss at saved epoch: 0.3136476254007064


In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision #import the MeanAveragePrecision class from the torchmetrics.detection.mean_ap module to compute the mean average precision (mAP) metric for object detection tasks

print("TorchMetrics imported successfully.")

TorchMetrics imported successfully.


In [ ]:
device = torch.device("cpu") # set the device to CPU for evaluation and inference
print("Device:", device)


from pathlib import Path 
import torch

PROJECT_ROOT = Path( # define the project root directory
    r"C:\Users\megdo\Desktop\underwater-object-detection"
)

RUN_DIR = ( # define the run directory for storing results and checkpoints
    PROJECT_ROOT
    / "results"
    / "faster_rcnn"
    / "duo_faster_rcnn_baseline"
)

BEST_CHECKPOINT = ( # define the path for the best checkpoint file within the run directory
    RUN_DIR
    / "checkpoint_epoch_05.pth"
)

print("Checkpoint exists:", BEST_CHECKPOINT.exists())
print(BEST_CHECKPOINT)


Device: cpu
Checkpoint exists: True
C:\Users\megdo\Desktop\underwater-object-detection\results\faster_rcnn\duo_faster_rcnn_baseline\checkpoint_epoch_05.pth


In [ ]:
from torchvision.models.detection import ( # import the necessary classes for creating a Faster R-CNN model
    FasterRCNN_ResNet50_FPN_Weights, # import the FasterRCNN_ResNet50_FPN_Weights class from the torchvision.models.detection module to access pre-trained weights for the Faster R-CNN model with ResNet50 FPN backbone
    fasterrcnn_resnet50_fpn, # import the fasterrcnn_resnet50_fpn function from the torchvision.models.detection module to create a Faster R-CNN model with ResNet50 FPN backbone
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor #import the FastRCNNPredictor class from the torchvision.models.detection.faster_rcnn module to customize the output layer of the Faster R-CNN model

weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT # load the default weights for the Faster R-CNN model with ResNet50 FPN backbone

model = fasterrcnn_resnet50_fpn( # create a Faster R-CNN model with ResNet50 FPN backbone using the specified weights
    weights=weights,
    min_size=416,
    max_size=416,
)

num_classes = 5 # background plus four DUO classes

in_features = ( # get the number of input features for the classifier layer of the Faster R-CNN model by accessing the in_features attribute of the cls_score layer of the box_predictor in the roi_heads of the model
    model.roi_heads
    .box_predictor
    .cls_score
    .in_features
)

model.roi_heads.box_predictor = FastRCNNPredictor( # replace the existing box_predictor of the Faster R-CNN model with a new FastRCNNPredictor that has the specified number of input features and output classes
    in_features,
    num_classes,
)

model = model.to(device) # move the model to the specified device (CPU in this case) for training and inference

In [ ]:
best_checkpoint = torch.load( # load the best checkpoint from the specified path
    BEST_CHECKPOINT,
    map_location=device, # load the checkpoint to the specified device (CPU in this case)
)

model.load_state_dict( # load the model state from the best checkpoint
    best_checkpoint["model_state_dict"]
)

model.eval() # set the model to evaluation mode, which disables certain layers like dropout and batch normalization that behave differently during training and inference

print(
    "Loaded checkpoint from epoch:",
    best_checkpoint["completed_epoch"],
)

Loaded checkpoint from epoch: 5


In [ ]:
import json #import the json module to work with JSON data, which is used for loading and parsing the COCO annotations in the dataset
import time
import torch

from collections import defaultdict 
from pathlib import Path

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F #import the functional module from torchvision.transforms to apply various image transformations, such as converting images to tensors, which is used in the DUOCocoDataset class for preprocessing the images
from torchmetrics.detection.mean_ap import MeanAveragePrecision #import the MeanAveragePrecision class from the torchmetrics.detection.mean_ap module to compute the mean average precision (mAP) metric for object detection tasks


PROJECT_ROOT = Path(
    r"C:\Users\megdo\Desktop\underwater-object-detection"
)

ANNOTATION_DIR = ( # define the path to the directory containing the COCO annotations for the DUO dataset
    PROJECT_ROOT
    / "data"
    / "processed"
    / "DUO_coco"
    / "annotations"
)

TEST_IMAGE_DIR = ( # define the path to the directory containing the test images for the DUO dataset
    PROJECT_ROOT
    / "data"
    / "raw"
    / "DUO"
    / "images"
    / "test"
)

TEST_JSON = ( # define the path to the JSON file containing the COCO annotations for the test set of the DUO dataset
    ANNOTATION_DIR
    / "instances_test_clean.json"
)


class DUOCocoDataset(Dataset): # define a custom dataset class for the DUO dataset in COCO format, which inherits from the PyTorch Dataset class
    def __init__(self, image_dir, annotation_file):
        self.image_dir = Path(image_dir)

        with Path(annotation_file).open(
            "r",
            encoding="utf-8",
        ) as file:
            self.coco = json.load(file)

        self.images = self.coco["images"]

        self.annotations_by_image = defaultdict(list)

        for annotation in self.coco["annotations"]: # iterate through the annotations in the COCO dataset and group them by image ID using a defaultdict of lists, which allows for easy retrieval of annotations for each image
            self.annotations_by_image[
                annotation["image_id"]
            ].append(annotation)

        category_ids = sorted(
            category["id"]
            for category in self.coco["categories"]
        )

        self.category_to_label = {
            category_id: index + 1
            for index, category_id in enumerate(category_ids)
        }

    def __len__(self): # return the number of images in the dataset, which is determined by the length of the self.images list
        return len(self.images)

    def __getitem__(self, index):
        image_record = self.images[index]

        image_path = (
            self.image_dir
            / image_record["file_name"]
        )

        image = Image.open(image_path).convert("RGB") # open the image file at the specified path and convert it to RGB format to ensure consistency in the number of channels across all images
        image = F.to_tensor(image)

        boxes = []
        labels = []

        annotations = self.annotations_by_image.get(
            image_record["id"],
            [],
        )

        for annotation in annotations:
            x, y, width, height = annotation["bbox"]

            x_min = x
            y_min = y
            x_max = x + width
            y_max = y + height

            if x_max <= x_min or y_max <= y_min:
                continue

            boxes.append(
                [x_min, y_min, x_max, y_max]
            )

            labels.append(
                self.category_to_label[
                    annotation["category_id"]
                ]
            )

        target = { # create a target dictionary containing the bounding boxes, labels, and image ID for the current image, which will be used for training and evaluation
            "boxes": torch.as_tensor(
                boxes,
                dtype=torch.float32,
            ).reshape(-1, 4),
            "labels": torch.as_tensor(
                labels,
                dtype=torch.int64,
            ),
            "image_id": torch.tensor(
                image_record["id"],
                dtype=torch.int64,
            ),
        }

        return image, target # return the preprocessed image tensor and the corresponding target dictionary for the current index, which will be used by the DataLoader to create batches for training and evaluation


def collate_fn(batch): # define a custom collate function for the DataLoader, which takes a batch of samples and returns a tuple containing the images and targets, allowing for proper batching of variable-length targets in object detection tasks
    return tuple(zip(*batch))


test_dataset = DUOCocoDataset( # create an instance of the DUOCocoDataset class for the test set
    TEST_IMAGE_DIR,
    TEST_JSON,
)

test_loader = DataLoader( # create a DataLoader for the test dataset, which will handle batching, shuffling, and parallel data loading for efficient evaluation of the model
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_fn,
)

print("Test images:", len(test_dataset))
print("Test batches:", len(test_loader))


metric = MeanAveragePrecision( # create an instance of the MeanAveragePrecision class to compute the mean average precision (mAP) metric for object detection tasks, which will be used to evaluate the performance of the trained Faster R-CNN model on the test dataset
    box_format="xyxy",
    iou_type="bbox",
    class_metrics=True,
)

model.eval() # set the model to evaluation mode, which disables certain layers like dropout and batch normalization that behave differently during training and inference

total_inference_seconds = 0.0 # initialize a variable to keep track of the total inference time across all test images
number_of_test_images = 0 # initialize a variable to keep track of the total number of test images processed during evaluation

with torch.no_grad(): # disable gradient computation to reduce memory usage and speed up inference during evaluation, as gradients are not needed for the forward pass when evaluating the model
    for batch_number, (images, targets) in enumerate(
        test_loader,
        start=1,
    ):
        images = [
            image.to(device)
            for image in images
        ]

        start_time = time.perf_counter()

        predictions = model(images)

        total_inference_seconds += (
            time.perf_counter() - start_time
        )

        number_of_test_images += len(images) # update the total number of test images processed by adding the number of images in the current batch

        metric_predictions = [ # create a list of dictionaries containing the predicted bounding boxes, scores, and labels for each image in the batch, which will be used to compute the mean average precision (mAP) metric
            {
                "boxes": prediction["boxes"].cpu(),
                "scores": prediction["scores"].cpu(),
                "labels": prediction["labels"].cpu(),
            }
            for prediction in predictions
        ]

        metric_targets = [ #    create a list of dictionaries containing the ground truth bounding boxes and labels for each image in the batch, which will be used to compute the mean average precision (mAP) metric
            {
                "boxes": target["boxes"].cpu(),
                "labels": target["labels"].cpu(),
            }
            for target in targets
        ]

        metric.update( # update the MeanAveragePrecision metric with the predicted and ground truth bounding boxes and labels for the current batch, allowing for the computation of the mean average precision (mAP) metric across all test images
            metric_predictions,
            metric_targets,
        )

        if batch_number == 1 or batch_number % 100 == 0:
            print(
                f"Test batch "
                f"{batch_number}/{len(test_loader)}"
            )


test_results = metric.compute() # compute the final mean average precision (mAP) metric and other related metrics, such as mAP@0.5 and mAP@0.5:0.95, based on the accumulated predictions and ground truth data across all test images

average_inference_seconds = ( # calculate the average inference time per image
    total_inference_seconds
    / number_of_test_images
)

approximate_fps = ( # calculate the approximate frames per second (FPS) based on the average inference time per image
    1 / average_inference_seconds
)

print("\nTest evaluation completed.")
print(
    "mAP@0.5:0.95:",
    float(test_results["map"]),
)
print(
    "mAP@0.5:",
    float(test_results["map_50"]),
)
print(
    "Average inference time:",
    f"{average_inference_seconds * 1000:.2f} ms/image",
)
print(
    "Approximate FPS:",
    f"{approximate_fps:.2f}",
)
print(
    "Per-class AP:",
    test_results["map_per_class"],
)
print(
    "Class labels:",
    test_results["classes"],
)

Test images: 1111
Test batches: 1111
Test batch 1/1111
Test batch 100/1111
Test batch 200/1111
Test batch 300/1111
Test batch 400/1111
Test batch 500/1111
Test batch 600/1111
Test batch 700/1111
Test batch 800/1111
Test batch 900/1111
Test batch 1000/1111
Test batch 1100/1111

Test evaluation completed.
mAP@0.5:0.95: 0.4720912575721741
mAP@0.5: 0.6949123740196228
Average inference time: 530.85 ms/image
Approximate FPS: 1.88
Per-class AP: tensor([0.4561, 0.6190, 0.2695, 0.5437])
Class labels: tensor([1, 2, 3, 4], dtype=torch.int32)


In [ ]:
import csv #import the csv module to work with CSV files, which will be used to save the test evaluation results in a structured format for further analysis and reporting

RESULTS_CSV = RUN_DIR / "faster_rcnn_test_results.csv" # define the path for the CSV file where the test evaluation results will be saved, which is located in the run directory and named "faster_rcnn_test_results.csv"

class_names = [ # define a list of class names corresponding to the DUO dataset, which will be used to label the per-class average precision (AP) values in the CSV file
    "holothurian",
    "echinus",
    "scallop",
    "starfish",
]

with RESULTS_CSV.open( #open the CSV file for writing, creating it if it doesn't exist, and specifying the encoding and newline handling to ensure proper formatting of the CSV file
    "w",
    newline="",
    encoding="utf-8",
) as file:
    writer = csv.writer(file)

    writer.writerow( #write the header row for the CSV file, which includes the column names "metric" and "value" to indicate the type of metric and its corresponding value
        [
            "metric",
            "value",
        ]
    )

    writer.writerow( #write the mAP@0.5:0.95 value to the CSV file, which represents the mean average precision across different intersection over union (IoU) thresholds, providing a comprehensive evaluation of the model's performance
        [
            "mAP@0.5:0.95",
            float(test_results["map"]),
        ]
    )

    writer.writerow( #write the mAP@0.5 value to the CSV file, which represents the mean average precision at a specific IoU threshold of 0.5, providing a more lenient evaluation of the model's performance
        [
            "mAP@0.5",
            float(test_results["map_50"]),
        ]
    )

    writer.writerow( #write the average inference time in milliseconds to the CSV file, which indicates the average time taken by the model to process a single image during inference, providing insight into the model's efficiency
        [
            "average_inference_ms",
            average_inference_seconds * 1000,
        ]
    )

    writer.writerow( #write the approximate frames per second (FPS) value to the CSV file, which indicates the model's inference speed, providing insight into its real-time performance
        [
            "approximate_fps",
            approximate_fps,
        ]
    )

    for class_name, ap_value in zip( #write the per-class average precision (AP) values to the CSV file, which provide insights into the model's performance for each individual class
        class_names,
        test_results["map_per_class"],
    ):
        writer.writerow(
            [
                f"{class_name}_AP@0.5:0.95",
                float(ap_value),
            ]
        )

print("Results saved to:", RESULTS_CSV) # test evaluation results  saved to the specified CSV file

Results saved to: C:\Users\megdo\Desktop\underwater-object-detection\results\faster_rcnn\duo_faster_rcnn_baseline\faster_rcnn_test_results.csv
